# ChurnGuard research prototype

**Group 49**

| BITS ID | Name | Contribution |
|---|---|---|
| 2025aa05877 | Vivek Kumar Aggarwal | Architecture and modularisation (100%) |
| 2025ab05013 | Nishant Choudhary | Reliability and code quality (100%) |
| 2025aa05544 | Sumanth T P | API and deployment readiness (100%) |
| 2025aa05974 | Ravi Raj Ladha | Quality assurance and model metrics (100%) |

This notebook contains the research prototype used in the Assignment II comparison. The production implementation is in `src/churnguard`.

In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
%matplotlib inline

In [2]:
df = pd.read_csv("../data/raw/telco_churn.csv")  # Prototype uses a notebook-relative path.
df.head()

,customer_id,tenure_months,monthly_charges,total_charges,support_tickets_6m,avg_monthly_gb,contract_type,internet_service,payment_method,tech_support,paperless_billing,churn
0,CUST-000001,60,37.91,2262.99,1,4.0,Month-to-month,No,Mailed check,Yes,Yes,1
1,CUST-000002,38,63.57,2385.51,0,11.6,Month-to-month,Fiber optic,Credit card,Yes,Yes,0
2,CUST-000003,61,63.92,3825.53,0,30.2,Two year,DSL,Bank transfer,Yes,Yes,0
3,CUST-000004,48,81.47,4117.85,1,30.0,Month-to-month,Fiber optic,Credit card,No,Yes,0
4,CUST-000005,7,59.85,426.08,0,18.2,One year,DSL,Electronic check,No,No,1


In [3]:
df.shape, df.churn.mean()

((5000, 12), np.float64(0.2666))

In [4]:
df.isnull().sum()

customer_id            0
tenure_months          0
monthly_charges        0
total_charges         60
support_tickets_6m     0
avg_monthly_gb        30
contract_type          0
internet_service       0
payment_method         0
tech_support           0
paperless_billing      0
churn                  0
dtype: int64

In [5]:
# Prototype shortcut: replace all missing values with zero.
df = df.fillna(0)

In [6]:
# Expand categorical columns for the standalone prototype model.
X = pd.get_dummies(df.drop(['churn', 'customer_id'], axis=1))
y = df.churn

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)  # Prototype does not fix a seed.

In [8]:
m = RandomForestClassifier(n_estimators=100)
m.fit(X_train, y_train)
p = m.predict_proba(X_test)[:, 1]
print(accuracy_score(y_test, m.predict(X_test)))
print(roc_auc_score(y_test, p))

0.766
0.7598844820947247


In [9]:
# Evaluate a derived spend-per-month feature.
X_train['spm'] = X_train.total_charges / X_train.tenure_months
X_test['spm'] = X_test.total_charges / X_test.tenure_months
m.fit(X_train, y_train)
roc_auc_score(y_test, m.predict_proba(X_test)[:, 1])

0.7563188294185599

In [10]:
# Compare a depth-limited model.
m2 = RandomForestClassifier(n_estimators=100, max_depth=8)
m2.fit(X_train, y_train)
roc_auc_score(y_test, m2.predict_proba(X_test)[:, 1])

0.7663919907585678

In [11]:
import pickle
pickle.dump(m2, open('model.pkl', 'wb'))  # Prototype persists only the estimator.